# 03 — Inference with Ollama (Local LLMs)

**Goal**: Use your locally-running Ollama models for text generation.

Ollama runs models like `llama3.2`, `mistral`, `phi3` entirely on your machine.

In [ ]:
import requests
import json

# Check if Ollama is running
try:
    r = requests.get("http://localhost:11434/api/tags")
    models = r.json()["models"]
    print(f"Ollama running. Available models:")
    for m in models:
        print(f"  - {m['name']} ({m['size'] / 1e9:.1f} GB)")
except:
    print("ERROR: Ollama not running! Start with: ollama serve")

## 1. Basic Text Generation

The `/api/generate` endpoint sends a prompt and gets a response.

In [ ]:
def ollama_generate(prompt, model="llama3.2", **kwargs):
    """Send a prompt to Ollama and get response."""
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": False,
        **kwargs
    })
    return resp.json()["response"]

response = ollama_generate("Explain what a vector database is in 2 sentences.")
print(response)

## 2. System Prompts

You can set a system message that defines the model's behavior.

In [ ]:
def ollama_chat(system, user, model="llama3.2"):
    """Chat with system prompt."""
    resp = requests.post("http://localhost:11434/api/chat", json={
        "model": model,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ],
        "stream": False
    })
    return resp.json()["message"]["content"]

response = ollama_chat(
    system="You are a helpful tutor. Explain concepts like I'm 12 years old.",
    user="How does machine learning work?"
)
print(response)

## 3. Controlling Generation Parameters

In [ ]:
prompt = "Write a short poem about programming"

print("=" * 60)
print("TEMPERATURE 0.1 (deterministic)")
print("=" * 60)
print(ollama_generate(prompt, temperature=0.1))

print("\n" + "=" * 60)
print("TEMPERATURE 0.8 (creative)")
print("=" * 60)
print(ollama_generate(prompt, temperature=0.8))

print("\n" + "=" * 60)
print("WITH TOP_P=0.3 (focused)")
print("=" * 60)
print(ollama_generate(prompt, temperature=0.7, top_p=0.3))

## 4. Streaming (Real-Time Output)

Instead of waiting for the full response, we can stream tokens as they're generated.

In [ ]:
def ollama_stream(prompt, model="llama3.2"):
    """Stream tokens as they're generated."""
    resp = requests.post("http://localhost:11434/api/generate", json={
        "model": model,
        "prompt": prompt,
        "stream": True
    }, stream=True)
    
    full_response = ""
    for line in resp.iter_lines():
        if line:
            data = json.loads(line)
            if "response" in data:
                full_response += data["response"]
                # Print in real-time without newline
                print(data["response"], end="", flush=True)
            if data.get("done"):
                break
    print()
    return full_response

print("Streaming response:\n")
result = ollama_stream("Count from 1 to 5 with a short pause between each.")

## 5. Comparing Models

Different Ollama models have different strengths.

In [ ]:
prompt = "What are the key differences between RAG and fine-tuning?"

models_to_try = ["llama3.2", "phi3", "mistral"]

for m in models_to_try:
    try:
        print(f"\n{'='*60}")
        print(f"MODEL: {m}")
        print('='*60)
        resp = ollama_generate(prompt, model=m, temperature=0.3)
        print(resp[:300] + "..." if len(resp) > 300 else resp)
    except Exception as e:
        print(f"Model {m} not available: {e}")

## 6. Token Counting (Important for Context Windows)

Ollama exposes token counts in its responses.

In [ ]:
long_prompt = "Write a detailed explanation of the transformer architecture." * 5

resp = requests.post("http://localhost:11434/api/generate", json={
    "model": "llama3.2",
    "prompt": long_prompt,
    "stream": False
})

data = resp.json()
print(f"Prompt tokens: {data['prompt_eval_count']}")
print(f"Response tokens: {data['eval_count']}")
print(f"Total duration: {data['total_duration'] / 1e9:.2f}s")

## Key Takeaways

1. **Ollama** = your private, free, local LLM server
2. Use **`/api/generate`** for simple prompts, **`/api/chat`** for chat with system prompts
3. **Temperature** and **top_p** control creativity
4. **Streaming** is better UX (tokens appear as generated)
5. Different models have different strengths — test multiple
6. Watch your **context window** (prompt + response tokens)

**Next**: Master prompt engineering techniques →